# Time Travel Rewind — GRPO Training with Unsloth + TRL

Trains a model to play the **Time Travel Rewind** environment using GRPO.

The environment runs on HuggingFace Spaces. This notebook:
1. Installs Unsloth + TRL
2. Loads `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit with Unsloth
3. Attaches LoRA
4. Runs GRPO where rewards come from the live OpenEnv server

**Required**: A GPU runtime (T4 or better). Go to Runtime → Change runtime type → T4 GPU.

In [ ]:
# Install Unsloth (nightly for Colab T4/A100 compatibility)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install 'openenv-core[core]==0.2.1' datasets

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
HF_SPACE_URL = "https://<your-hf-username>-timetravel.hf.space"  # ← change this
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN  = 2048
LORA_RANK    = 16
BUDGET       = 12   # must match server setting
NUM_EPISODES = 200  # episodes to collect per GRPO step
GRPO_STEPS   = 100

## 1. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect (bfloat16 on A100, float16 on T4)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded with LoRA.")

## 2. Episode Rollout via OpenEnv

In [ ]:
import json
import requests

PASSCODES = ("AURORA-314", "CINDER-271", "EMBER-907", "NOVA-553")


def env_reset(session_id: str = "0") -> str:
    """Reset the environment and return the task prompt text."""
    r = requests.post(f"{HF_SPACE_URL}/reset", json={"session_id": session_id})
    r.raise_for_status()
    data = r.json()
    return data["observation"]["message"]


def env_step(action_json: str, session_id: str = "0"):
    """Send one action, return (obs_message, temporal_note, reward, done)."""
    payload = {"content": action_json, "session_id": session_id}
    r = requests.post(f"{HF_SPACE_URL}/step", json=payload)
    r.raise_for_status()
    data = r.json()
    obs  = data["observation"]
    return (
        obs["message"],
        obs.get("temporal_note"),
        data.get("reward", 0.0),
        data.get("done", False),
    )


def build_messages(system: str, history: list) -> list:
    """Build a chat-format message list for the tokenizer."""
    msgs = [{"role": "system", "content": system}]
    msgs.extend(history)
    return msgs


SYSTEM_PROMPT = (
    "You are an agent navigating a time-travel puzzle. "
    "Always reply with exactly one JSON object: "
    '{"thinking": "...", "action": "...", "args": {...}}'
)


def generate_action(history: list) -> str:
    """Run the current model to generate the next JSON action."""
    FastLanguageModel.for_inference(model)
    messages = build_messages(SYSTEM_PROMPT, history)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def run_episode(episode_idx: int) -> dict:
    """Run one full episode and return (prompt, completion, reward) for GRPO."""
    sid = str(episode_idx)
    task_prompt = env_reset(sid)

    history = [{"role": "user", "content": task_prompt}]
    episode_text = task_prompt  # full text of the episode
    final_reward = 0.0

    for _ in range(BUDGET + 5):  # +5 slack for protocol violations
        action_str = generate_action(history)
        history.append({"role": "assistant", "content": action_str})
        episode_text += "\n" + action_str

        obs_msg, temporal_note, reward, done = env_step(action_str, sid)

        # Compose environment response (include temporal note if present)
        env_reply = obs_msg
        if temporal_note:
            env_reply = temporal_note + "\n" + obs_msg

        history.append({"role": "user", "content": env_reply})
        episode_text += "\n" + env_reply

        if done:
            final_reward = reward
            break

    return {
        "prompt": task_prompt,
        "completion": episode_text[len(task_prompt):],
        "reward": final_reward,
    }


# Quick smoke test
ep = run_episode(0)
print(f"Smoke test — reward: {ep['reward']:.3f}")
print("Episode snippet:", ep['completion'][:300])

## 3. GRPO Training

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

# ── Reward function for GRPOTrainer ────────────────────────────────────────────
# GRPOTrainer calls reward_funcs with (prompts, completions) batches.
# We run each completion through the live environment to get the real reward.

def timetravel_reward(prompts: list[str], completions: list[str], **kwargs) -> list[float]:
    """Score each completion by running it through the OpenEnv environment."""
    rewards = []
    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        sid = f"grpo_{i}"
        env_reset(sid)
        # Step through each model turn in the completion
        turns = [t.strip() for t in completion.split("\n") if t.strip()]
        final_reward = 0.0
        for turn in turns:
            # Only send lines that look like JSON actions
            if turn.startswith("{"):
                _, _, reward, done = env_step(turn, sid)
                if done:
                    final_reward = reward
                    break
        rewards.append(final_reward)
    return rewards


# ── Build a small seed dataset (GRPO needs a dataset to iterate over) ──────────
PASSCODES = ("AURORA-314", "CINDER-271", "EMBER-907", "NOVA-553")

TASK_PROMPT = (
    f"Temporal gate mission. Budget: {BUDGET} actions.\n"
    "Map: start -> door -> path-1 -> path-2 -> sign\n"
    "You start at `start`. The door is locked and requires a passcode.\n"
    "The passcode can only be read at `sign`.\n\n"
    "Output exactly one JSON object per turn:\n"
    '{"thinking":"...", "action":"...", "args":{...}}\n\n'
    "Actions: move_forward, move_back, read_sign, open_door, branch, abandon\n"
    "branch args: {\"ago\": <int>, \"instruction\": \"...\"}\n"
    "open_door args: {\"passcode\": \"...\"}\n\n"
    "Intended strategy: reach sign, read passcode, branch back near door, open door."
)

seed_data = [
    {"prompt": TASK_PROMPT, "answer": pc}
    for pc in PASSCODES * 30  # 120 examples, cycling through passcodes
]
train_dataset = Dataset.from_list(seed_data)

# ── GRPO config ────────────────────────────────────────────────────────────────
FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir="timetravel-grpo",
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=GRPO_STEPS,
    max_completion_length=512,
    num_generations=4,          # rollouts per prompt
    temperature=0.7,
    beta=0.001,                 # KL penalty weight
    logging_steps=5,
    save_steps=50,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    report_to="none",           # swap to "wandb" if you set up W&B
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[timetravel_reward],
    args=grpo_config,
    train_dataset=train_dataset,
)

print("Starting GRPO training...")
trainer.train()

## 4. Save & Push to HuggingFace Hub

In [ ]:
HF_USERNAME = "<your-hf-username>"   # ← change this
REPO_NAME   = "timetravel-grpo-qwen2.5-1.5b"

# Save merged 16-bit model (or just the LoRA adapter)
model.save_pretrained_merged(
    REPO_NAME,
    tokenizer,
    save_method="merged_16bit",
)

# Push to Hub
model.push_to_hub_merged(
    f"{HF_USERNAME}/{REPO_NAME}",
    tokenizer,
    save_method="merged_16bit",
    token="",   # ← paste your HF write token, or use: from huggingface_hub import login; login()
)
print(f"Pushed to https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

## 5. Quick Eval — Run a Sample Episode with Trained Model

In [ ]:
FastLanguageModel.for_inference(model)

ep = run_episode(episode_idx=99)
print(f"Final reward : {ep['reward']:.3f}")
print()
print(ep['completion'])